# Lean Agent Playground

Wires up a smolagents `CodeAgent` with a hand-picked list of tools, runs it, and saves the run logs to disk.

Requires `NEBIUS_API_KEY` or `TOKEN_FACTORY_API_KEY` in `.env`.

In [ ]:
from smolagents import CodeAgent, ToolCallingAgent, OpenAIServerModel

from lean_agent import get_settings, save_run
from lean_agent.tools import (
    caesar_cipher,
    compound_interest,
    fibonacci,
    get_current_time,
    is_palindrome,
    is_prime,
    reverse_words,
    roll_dice,
    temperature_convert,
    word_count,
)

settings = get_settings()
print(f"model:   {settings.model_id}")
print(f"log dir: {settings.log_dir}")

---

## 1. Build the agent

Three things to notice:

1. **`tools=[...]`** &mdash; an explicit list. Swap tools in/out to change what the agent can do.
2. **`max_steps`** &mdash; hard cap on the number of reasoning iterations. Useful guard against runaway loops.
3. **`instructions`** &mdash; short, high-priority guidance prepended to the system prompt. Keep it tight; it competes with the rest of smolagents' built-in prompt.

In [ ]:
model = OpenAIServerModel(
    model_id=settings.model_id,
    api_key=settings.api_key,
    api_base=settings.api_base,  # Token Factory by default; can be any OpenAI-compatible endpoint
)

tools = [
    get_current_time,
    word_count,
    fibonacci,
    compound_interest,
    is_prime,
    is_palindrome,
    temperature_convert,
    roll_dice,
    caesar_cipher,
    reverse_words,
]

agent = CodeAgent(
    tools=tools,
    model=model,
    max_steps=12,
    instructions="Prefer using a tool when one fits the question. Be concise.",
)

print(f"{len(tools)} tools wired up:")
for t in tools:
    print(f"  - {t.name}")

---

## 2. Run a prompt that exercises several tools

In [ ]:
task = (
    "Answer each of the following using the available tools:\n"
    "1. Is 'A man, a plan, a canal: Panama' a palindrome?\n"
    "2. Is 9973 prime?\n"
    "3. What is 100 degrees Fahrenheit in Celsius?\n"
    "4. What is the 15th Fibonacci number?\n"
    "Return a short numbered summary."
)

answer = agent.run(task)

print("\n--- final answer ---")
print(answer)

In [ ]:
print(answer)

---

## 3. Save the run logs

`save_run(agent, answer)` writes two files into a fresh per-run directory under `logs/`. Pass `run_id="my-experiment"` if you want to label this run; otherwise it auto-generates one.

- `run.json` &mdash; `{manifest, prompt, answer, logs}` with per-step + total token usage.
- `transcript.yaml` &mdash; sanitized `system / user / assistant / tool-*` linear chat view.

In [ ]:
run_dir = save_run(agent, answer)
print(f"logs saved to: {run_dir}")
for f in sorted(run_dir.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size} bytes)")